# Faruq-v3 — IGEM1 selector-feasibility audit

Validation-only, post-training, **tanpa inference ulang dan tanpa training**. Audit memakai object-event JSON seed42 yang sudah dibuat untuk membandingkan IGEM1 vs AF2 dan IGEM1 vs SAF1. Hasil ini GT-anchored diagnostic, bukan router deployable. Test tidak dibuka.

Gate statistik dibekukan sebelum hasil dibaca: confidence signal hanya SUPPORTED jika Wilson 95% lower bound untuk memilih expert yang benar > 50% dan zero-threshold switch memberi net correct delta positif terhadap IGEM1.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import importlib, json, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/multimodel-complementarity-audit'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
SRC = str(REPO / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
importlib.invalidate_caches()
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], cwd=REPO, text=True).strip())
print('coffee_detector import: OK')

In [ ]:
BASE = 'experiments/faruq-v3-multimodel-complementarity-seed42-v1/events'
REQUIRED = (
    f'{BASE}/IGEM1_seed42_events.json',
    f'{BASE}/AF2_seed42_events.json',
    f'{BASE}/SAF1_seed42_events.json',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
IGEM = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
AF2 = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
SAF = require_project_artifact(PROJECT_ROOT, REQUIRED[2])
OUTPUT = PROJECT_ROOT / 'experiments/faruq-v3-selector-feasibility-seed42-v1/igem1_selector_feasibility.json'
print('IGEM:', IGEM)
print('AF2 :', AF2)
print('SAF1:', SAF)
print('OUT :', OUTPUT)

In [ ]:
command = [
    sys.executable, '-m', 'coffee_detector.analysis.selector_feasibility_from_events_v2',
    '--primary', str(IGEM),
    '--candidate', str(AF2),
    '--candidate', str(SAF),
    '--output', str(OUTPUT),
]
subprocess.run(command, cwd=REPO, check=True)
result = json.loads(OUTPUT.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
assert result['frozen_before_results'] is True
print('AUDIT SELESAI:', OUTPUT)

In [ ]:
import pandas as pd
from IPython.display import display

pair_rows = []
for pair in result['pairs']:
    pair_rows.append({
        'pair': f"{pair['primary']} vs {pair['candidate']}",
        'joint_match': pair['joint_matched_rate'],
        'disagreement_n': pair['disagreement_targets'],
        'disagreement_rate': pair['disagreement_rate_among_joint_matched'],
        'resolvable_n': pair['resolvable_disagreements_exactly_one_correct'],
        'resolvable_fraction': pair['resolvable_fraction_of_disagreements'],
        'primary_acc_disagree': pair['primary_accuracy_on_disagreements'],
        'candidate_acc_disagree': pair['candidate_accuracy_on_disagreements'],
        'higher_conf_acc_disagree': pair['higher_conf_accuracy_on_all_disagreements'],
        'higher_conf_picks_correct_expert': pair['higher_conf_correct_expert_rate_when_exactly_one_correct'],
        'wilson95_low': pair['higher_conf_correct_expert_wilson95_low'],
        'wilson95_high': pair['higher_conf_correct_expert_wilson95_high'],
        'oracle_acc_disagree': pair['oracle_accuracy_on_disagreements'],
        'zero_switch_net_delta': pair['zero_threshold_net_correct_delta_vs_primary'],
        'primary_conf_auc': pair['primary_self_awareness']['confidence_correctness_auc'],
        'candidate_conf_auc': pair['candidate_self_awareness']['confidence_correctness_auc'],
        'decision': pair['frozen_confidence_signal_gate']['decision'],
    })
fmt = {k:'{:.2%}' for k in ('joint_match','disagreement_rate','resolvable_fraction','primary_acc_disagree','candidate_acc_disagree','higher_conf_acc_disagree','higher_conf_picks_correct_expert','wilson95_low','wilson95_high','oracle_acc_disagree','primary_conf_auc','candidate_conf_auc')}
display(pd.DataFrame(pair_rows).style.format(fmt))

for pair in result['pairs']:
    print('\n===', pair['primary'], 'vs', pair['candidate'], '===')
    print('FROZEN DECISION:', pair['frozen_confidence_signal_gate']['decision'])
    print('CONFIDENCE GAP BINS')
    display(pd.DataFrame(pair['confidence_gap_bins']).style.format({
        'primary_accuracy':'{:.2%}',
        'candidate_accuracy':'{:.2%}',
        'higher_conf_accuracy_all_disagreements':'{:.2%}',
        'higher_conf_correct_expert_rate_when_resolvable':'{:.2%}',
        'oracle_accuracy':'{:.2%}',
    }))
    print('DESCRIPTIVE SWITCH THRESHOLD SWEEP — bukan frozen model-selection gate')
    display(pd.DataFrame(pair['candidate_switch_threshold_sweep']).style.format({
        'threshold':'{:.2f}', 'switch_rate':'{:.2%}', 'gt_aligned_object_accuracy':'{:.2%}'
    }))
print('Kirim tabel utama + gap bins + threshold sweep. Jangan membuka test.')